# Jour 2 · Mini-projet : analyser un réseau de neurones

> **Version corrigée** — les cellules de solution sont signalées.

## Objectifs

- entraîner un petit CNN avec un protocole reproductible
- interpréter classe, probabilités et confiance
- analyser les erreurs et tester la robustesse à une variation simple


## Mission

Nous allons mener une petite expérience complète de Deep Learning : entraîner un CNN, produire des probabilités, observer les erreurs et tester ce qui se passe lorsque les images changent légèrement.

L'objectif n'est pas de chercher le score maximal. Il est de relier toutes les notions de la journée dans une démarche reproductible.


![Boucle question données modèle évaluation et itération](../../ressources/illustrations/jour_02/04_demarche_experimentale.png)

*Un projet progresse par hypothèses vérifiables, pas en empilant des changements au hasard.*


## Une expérience reproductible

`set_random_seed(42)` fixe plusieurs sources de hasard utilisées par TensorFlow. Cela aide deux exécutions à rester comparables, sans garantir une identité absolue sur toutes les machines et tous les matériels.

Le code reprend les étapes vues précédemment : chargement, conversion en `float32`, normalisation, séparation stratifiée, ajout du canal, création du CNN, compilation et entraînement.

Répéter une expérience ne signifie pas seulement relancer le notebook. Il faut conserver les données, le découpage, l’architecture, les hyperparamètres et les versions logicielles qui peuvent influencer le résultat.


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass
plt.style.use("seaborn-v0_8-whitegrid")

digits = load_digits()
images = digits.images.astype("float32") / 16.0
labels = digits.target.astype("int64")
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=.25, stratify=labels, random_state=42)
X_train_cnn = X_train[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = cnn.fit(X_train_cnn, y_train, validation_split=.20, epochs=10, batch_size=32, verbose=0)
print("Entraînement terminé.")


## Une sortie probabiliste

Pour chaque image, la couche softmax produit dix valeurs dont la somme vaut environ 1. La classe prédite est celle qui possède la plus grande valeur. Nous appellerons cette valeur maximale **confiance du modèle**.

Cette confiance décrit la préférence interne du réseau entre ses classes. Elle ne constitue pas une garantie : un réseau peut être très confiant et se tromper.


![Deux distributions de confiance sur dix classes](../../ressources/illustrations/jour_02/04_prediction_confiance.png)

*La forme de la distribution aide à distinguer une décision nette d’une décision hésitante.*


![Confiance Softmax et calibration](../../ressources/illustrations/jour_02/04_confiance_calibration.png)

`probabilities.max(axis=1)` garde le score le plus élevé de chaque ligne ; `argmax(axis=1)` garde l’indice de ce score. Les deux informations répondent à des questions différentes : « quelle classe arrive première ? » et « avec quelle préférence interne ? ».

Un modèle est **bien calibré** si, parmi les prédictions annoncées autour de 80 %, environ 80 % sont effectivement correctes sur un grand nombre de cas comparables. Softmax ne garantit pas automatiquement cette propriété.

Il faut donc éviter de traduire une confiance individuelle de 90 % par « il y a exactement 90 % de chances que ce soit vrai ». Cette valeur reste un signal produit par le modèle et doit être vérifiée sur des données représentatives.


In [ ]:
probabilities = cnn.predict(X_test_cnn, verbose=0)
predictions = probabilities.argmax(axis=1)
confidence = probabilities.max(axis=1)
correct = predictions == y_test

results = pd.DataFrame({
    "réel": y_test,
    "prédit": predictions,
    "confiance": confidence,
    "correct": correct,
})
print("Accuracy test :", round(accuracy_score(y_test, predictions), 3))
print("Confiance moyenne si correct :", round(results.loc[results.correct, "confiance"].mean(), 3))
print("Confiance moyenne si erreur  :", round(results.loc[~results.correct, "confiance"].mean(), 3))
results.head()


![Étapes d’analyse des prédictions et des erreurs](../../ressources/illustrations/jour_02/04_analyser_erreurs.png)

*Les métriques indiquent combien ; les exemples montrent comment et dans quelles situations.*


### Donner une catégorie aux erreurs

Une bonne analyse ne se limite pas à afficher des images. On cherche des groupes explicables :

- confusion entre deux classes visuellement proches ;
- image peu lisible ou partiellement effacée ;
- cadrage ou orientation inhabituels ;
- exemple rare dans le train ;
- label potentiellement incorrect.

Cette **taxonomie d’erreurs** transforme une liste de cas ratés en pistes d’action : collecter certains exemples, corriger des labels, modifier la préparation ou revoir la question. Elle évite de changer l’architecture au hasard.


In [ ]:
cm = confusion_matrix(y_test, predictions)
pd.DataFrame(cm, index=[f"réel {i}" for i in range(10)], columns=[f"prédit {i}" for i in range(10)])


## Examiner les erreurs les plus confiantes

Une erreur peu confiante peut simplement correspondre à une image ambiguë. Une erreur très confiante est particulièrement instructive : le modèle a reconnu avec assurance un motif trompeur. Afficher ces images peut révéler des classes proches, des cas atypiques ou des labels discutables.


In [ ]:
wrong = np.flatnonzero(~correct)
wrong = wrong[np.argsort(confidence[wrong])[::-1]][:6]
fig, axes = plt.subplots(1, len(wrong), figsize=(12, 2.3))
axes = np.atleast_1d(axes)
for ax, index in zip(axes, wrong):
    ax.imshow(X_test[index], cmap="gray_r")
    ax.set_title(f"réel {y_test[index]}\nprédit {predictions[index]} ({confidence[index]:.0%})")
    ax.axis("off")
plt.tight_layout(); plt.show()


## Autoriser le modèle à ne pas décider

On peut fixer un seuil minimal de confiance et ne conserver que certaines prédictions. La **couverture** est la proportion d'images pour lesquelles le modèle accepte de répondre. Un seuil élevé réduit généralement la couverture, sans garantir l'absence d'erreurs.


![Couverture, qualité acceptée et test de robustesse](../../ressources/illustrations/jour_02/04_couverture_robustesse.png)

Si le seuil minimal vaut 0,80, une prédiction de confiance 0,72 est refusée. La **couverture** compte la proportion de toutes les images qui restent acceptées.

L’accuracy calculée uniquement sur les prédictions acceptées peut augmenter lorsque le seuil monte, mais ce n’est pas garanti. Il faut toujours présenter ensemble la qualité et la couverture : une accuracy parfaite sur 2 % des cas ne décrit pas un système utile pour les 98 % restants.

Un refus n’est pas nécessairement un échec. Il peut déclencher une vérification humaine ou une autre procédure. Le comportement prévu pour les cas refusés doit être défini avec le besoin métier.


## À vous de jouer — mesurer le compromis confiance-couverture

Pour les seuils 0,60, 0,80 et 0,90, calculez la couverture et l’accuracy uniquement sur les prédictions acceptées.


In [ ]:
rows=[]
for threshold in [.60,.80,.90]:
    accepted=confidence>=threshold
    rows.append({
        "seuil":threshold,
        "couverture":accepted.mean(),
        "accuracy_acceptée":accuracy_score(y_test[accepted],predictions[accepted]) if accepted.any() else np.nan,
    })
pd.DataFrame(rows).set_index("seuil").round(3)


## Tester une petite variation des données

Un modèle est évalué sur une distribution donnée. Si les images deviennent plus bruitées, plus sombres ou cadrées différemment, sa qualité peut diminuer. Cette expérience ne simule pas tous les changements possibles ; elle rend seulement visible la fragilité potentielle.


### Robustesse et décalage de distribution

La **robustesse** décrit la capacité à conserver un comportement acceptable lorsque les entrées subissent une variation plausible. Ajouter un bruit gaussien modifie aléatoirement les pixels ; `np.clip` ramène ensuite les valeurs dans l’intervalle autorisé `[0, 1]`.

Ce test est volontairement simple. Il ne prouve pas que le réseau résistera à toutes les situations réelles. Il montre comment comparer une condition de référence et une condition modifiée avec les mêmes métriques.

Si la performance chute, plusieurs réponses sont possibles : collecter des données plus représentatives, utiliser une augmentation adaptée, améliorer la qualité des entrées ou définir un mécanisme de refus. La bonne réponse dépend de la cause observée.


## À vous de jouer — ajouter du bruit aux images

Ajoutez un bruit gaussien d’écart-type 0,25 aux images du test, limitez les pixels entre 0 et 1, puis comparez accuracy et confiance moyenne.


In [ ]:
rng=np.random.default_rng(42)
X_noisy=np.clip(X_test+rng.normal(0,.25,X_test.shape),0,1).astype("float32")
noisy_probabilities=cnn.predict(X_noisy[...,np.newaxis],verbose=0)
noisy_predictions=noisy_probabilities.argmax(axis=1)
print("Accuracy originale :",round(accuracy_score(y_test,predictions),3))
print("Accuracy bruitée   :",round(accuracy_score(y_test,noisy_predictions),3))
print("Confiance moyenne originale :",round(confidence.mean(),3))
print("Confiance moyenne bruitée   :",round(noisy_probabilities.max(axis=1).mean(),3))


## Bilan du mini-projet

Nous avons séparé train et test, entraîné un CNN, lu sa sortie softmax, calculé une métrique globale puis affiché ses erreurs. Nous avons également vérifié qu'un seuil de confiance modifie la couverture et qu'un changement des entrées peut dégrader les résultats.

**À retenir :** un bon modèle n'est pas seulement un modèle qui obtient une accuracy élevée. Il faut comprendre les données utilisées, les erreurs commises et les situations dans lesquelles la conclusion devient incertaine.
